In [2]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
docs = (
    PyPDFLoader("../documents/Company_Policies.pdf").load() + 
    PyPDFLoader("../documents/Company_Profile.pdf").load() + 
    PyPDFLoader("../documents/Product_and_Pricing.pdf").load() 
)

In [6]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=150, 
).split_documents(docs)

In [7]:
len(chunks)

16

In [9]:
embeddings = OllamaEmbeddings(model='bge-m3')
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings, 
    persist_directory="../vector-database/selfrag1"
)

In [10]:
retriever = vector_store.as_retriever(search_kwargs={'k':4})

In [11]:
llm = ChatOllama(model="llama3.2", temperature=0)

In [12]:
#Graph state
class State(TypedDict): 
        question: str 
        need_retrieval: bool
        docs: List[Document]

        answer: str

In [13]:
class RetrieveDecision(BaseModel): 
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed ti answer reliably, else False,"
    )